In [1]:
from datetime import date

#####
##### パラメータの設定
##### 
valuation_date = date.today()
min_gr_sales = 10
min_diff_ordinary_profit_growth_rate = 20
max_expected_PER = 20
min_sales = 2500

In [2]:
#####
##### import objects
#####
from pathlib import Path
import os

CURRENT_DIR = Path(os.getcwd())
PJ_DIR = CURRENT_DIR.parent.parent
DATA_DIR = PJ_DIR / "data" 



import polars as pl
from wequant.data_processing import KessanPl, FinancequotePl, PricelistPl

ModuleNotFoundError: No module named 'wequant.data_processing'

In [3]:
# code, name, 売上高, 売上高伸率, 差分経常利益伸率

In [4]:
#
# KPLの加工
#
KPL = KessanPl()
KPL.with_columns_profit_rate()
KPL.with_columns_growth_rate()
KPL.with_columns_diff_growth_rate()
KPL.with_columns_columns_ratio(
    "diff_ordinary_profit_growth_rate",
    "pr_ordinary_profit",
    "diff/pr"
)
KPL.with_columns_company_name()
df = KPL.get_latest_quater_settlements(valuation_date=valuation_date)
# 列の選択
select_cols = [
    "code",
    "name",
    "settlement_date",
    "announcement_date",
    "sales",
    "ordinary_profit",
    "pr_ordinary_profit",
    "gr_sales",
    "diff_ordinary_profit_growth_rate",
    "diff/pr"
]
df = df.select(select_cols)

target_df = df

In [5]:
#
# FPL.dfを取得してtarget_dfに連結
#
FPL = FinancequotePl()
df = FPL.get_finance_quotes(valuation_date)

# select
select_cols = [
    "code",
    "expected_PER",
    "actual_PBR"
]
df = df.select(select_cols)

# 連結
target_df = target_df.join(df, on=["code"], how="left")

In [6]:
#####
##### パラメータの設定
#####
# 売上高が伸び、経常利益率がそれ以上に伸びている銘柄
valuation_date = date.today()
min_gr_sales = 10
min_diff_ordinary_profit_growth_rate = 30
max_expected_PER = 15
min_sales = 2500
min_diff_ordinary_profit_growth_rate_gr_sales_ratio = 2

#
# filter
#
df = target_df
df = (df
      .filter(pl.col("gr_sales") >= min_gr_sales)
      .filter(pl.col("diff_ordinary_profit_growth_rate") >= min_diff_ordinary_profit_growth_rate)
      .filter(pl.col("expected_PER") < max_expected_PER)
      .filter(pl.col("sales") >= min_sales)
      .filter(pl.col("diff_ordinary_profit_growth_rate")/pl.col("gr_sales") >= min_diff_ordinary_profit_growth_rate_gr_sales_ratio)
)

#
# sort
#
df = df.sort(by=["diff/pr"], descending=[True])

result_df = df

In [7]:
result_df[10:20]

code,name,settlement_date,announcement_date,sales,ordinary_profit,pr_ordinary_profit,gr_sales,diff_ordinary_profit_growth_rate,diff/pr,expected_PER,actual_PBR
i64,str,date,date,i64,i64,f64,f64,f64,f64,f64,f64


In [8]:
# 2026/1/18
# 7172	"ジャパンインベストメントアドバイザー"	:オペレーションリース
# 6349 小森コーポレーション: オフセット印刷、市場はゆるやかな成長を予測、紙幣等セキュリティ印刷、海外69、配当4%超、per12.3、21年起点に連続増収、4Q重、高配当
# 4506	"住友ファーマ": 業績急回復、無配
# 6533	"Orchestra HD": 利益急改善、前四半期発表連続ストップ高　per13.5 ソフトウェア検証好調
# 4838	"スペースシャワーSKIYAKI HD": 利益急改善、フェス。夏重、訪日銘柄？？メイドカフェ、アニメフェス、コスプレ、焼酎フェスなど。範囲拡大？
# 2354	"YE DIGITAL": 安川持ち分。iot強化。利益回復
#6351	"鶴見製作所": 水中ポンプ世界首位、グローバル展開活発、長く成長、4Q重

